In [1]:
%load_ext autoreload
# autoreload 2：改了 benchmark/ 下的文件不用重启 kernel
%autoreload 2

import os
os.chdir(os.path.expanduser("~/projects/LLM/hw2"))   # benchmark 包以 hw2/ 为根

import torch
import pandas as pd
from benchmark import attention_sweep, bench_attention
from cs336_basics.model import scaled_dot_product_attention as sdpa   # §4.1 的被测对象

torch.backends.cuda.matmul.allow_tf32 = False        # fp32 基准：不偷偷走 tensor core
torch.backends.cudnn.allow_tf32 = False

PROF = "profiles"                       # nsys 报告、显存快照（gitignored）
def assets(sec):                        # 每节的落盘目录 notes/assets/s<sec>；各节开头 OUT = assets(n)
    d = f"notes/assets/s{sec}"; os.makedirs(d, exist_ok=True); return d

print(torch.cuda.get_device_name(0), torch.__version__)

NVIDIA GeForce RTX 5090 2.11.0+cu130


## 4 GPU Kernels

PDF §4。编号按 problem 顺序（4.1 `pytorch_attention` = PDF 4.1.1，4.2 `torch_compile` = PDF 4.2，
4.3–4.5 三题在 PDF 里都归在 4.2.2 下）。

In [2]:
OUT = assets(4)

### 4.1 `pytorch_attention`

被测的是 hw1 的 `scaled_dot_product_attention(q, k, v, mask=None)`，
输入 `[batch, seq, d]` —— batch 固定 8，**去掉 head 维**（题面 (i)）。

每个 (d, seq) 格子量五个数：100 次前向 ms、反向开始前的 `memory_allocated()`、100 次反向 ms，外加前向峰值和全程峰值 `max_memory_allocated()`；
OOM 的格子记 `status`，不中断 sweep。

In [3]:
# 实验网格（题面 (ii)）：4 × 5 = 20 格。大格子预期 OOM —— 那正是题目要的答案
D_MODELS = [16, 32, 64, 128]
SEQ_LENS = [256, 1024, 4096, 8192, 16384]

df = attention_sweep(sdpa, D_MODELS, SEQ_LENS, f"{OUT}/attention_fp32.md")
df

[20/20] d=128 seq=16384


,d,seq,fwd_ms,bwd_ms,mem_before_bwd_gib,peak_fwd_gib,peak_gib,status
0,16,256,0.078890,0.210671,0.020287,0.024078,0.028466,ok
1,16,1024,0.179895,0.497667,0.080414,0.142456,0.206880,ok
2,16,4096,4.672737,11.614525,1.024048,2.022217,3.029908,ok
3,16,8192,18.526315,45.488487,4.032227,8.028564,12.043946,ok
4,16,16384,NaN,NaN,NaN,NaN,NaN,OOM@warmup
5,32,256,0.073607,0.246052,0.020775,0.024445,0.029321,ok
6,32,1024,0.172775,0.523051,0.082367,0.143921,0.210298,ok
7,32,4096,4.709956,11.601442,1.031860,2.028076,3.043580,ok
8,32,8192,18.720092,47.258119,4.047852,8.040283,12.071290,ok
9,32,16384,NaN,NaN,NaN,NaN,NaN,OOM@warmup


#### 表 4.1-1 / 4.1-2

`assets/s4/tables_4_1.py`：表 4.1-1 是 sweep 的透视；`forward_steps(8192)` 逐步跑一遍 sdpa 前向、每步读 `memory_allocated()`，
给出表 4.1-2「前向峰值 4 份」的依据，seq 16384 列按 ×4 外推（它本身跑不到第 4 步）。

In [4]:
import runpy
T = runpy.run_path(f"{OUT}/tables_4_1.py")
t1, t2 = T["pivot_tables"](df)
steps = T["forward_steps"](8192); steps["seq 16384 外推 GiB"] = steps.iloc[:, 2] * 4
display(t1, t2, steps)

seq,256,1024,4096,8192,16384
d,,,,,
16,0.08 / 0.21,0.18 / 0.50,4.67 / 11.61,18.53 / 45.49,OOM
32,0.07 / 0.25,0.17 / 0.52,4.71 / 11.60,18.72 / 47.26,OOM
64,0.08 / 0.25,0.19 / 0.54,5.05 / 12.24,20.15 / 48.01,OOM
128,0.08 / 0.25,0.26 / 0.69,6.09 / 14.14,24.28 / 55.30,OOM


seq,256,1024,4096,8192,16384
反向前 memory_allocated,0.020,0.080,1.024,4.032,OOM（估 16.0）
前向峰值,0.024,0.142,2.022,8.029,OOM（估 32.0）
前向+反向峰值,0.028,0.207,3.030,12.044,OOM（估 48.0）


,前向步骤,新增张量,seq 8192 活着的 GiB,seq 16384 外推 GiB
0,S = QKᵀ/√d,S,2.000000,8.000000
1,x − max,x_shifted,4.000732,16.002930
2,exp(x_shifted),nume,6.000732,24.002930
3,P = nume / sum,P,8.000977,32.003906
4,函数返回（释放 S、x_shifted）,,4.004639,16.018555


#### 两个验证

1. seq 16384 炸的时候显存里已经有多少——预期 3 份 = 24 GiB，第 4 份申请失败。
2. 换成融合的 `torch.softmax`（只有 S、P 两份），同一格前向能不能过——预期能，峰值 16 GiB。

In [5]:
GiB = 2**30
Q, K, V = (torch.randn(8, 16384, 16, device="cuda", requires_grad=True) for _ in range(3))

# 1. hw1 手写 softmax：炸在第 4 份
torch.cuda.reset_peak_memory_stats()
try:
    O = sdpa(Q, K, V); torch.cuda.synchronize(); print("hw1 版前向过了？", torch.cuda.max_memory_allocated() / GiB)
except torch.cuda.OutOfMemoryError:
    print(f"hw1 版前向 OOM：炸时已分配 {torch.cuda.memory_allocated() / GiB:.1f} GiB")
torch.cuda.empty_cache()

# 2. 融合 softmax：只有 S 和 P
import math
from einops import einsum
torch.cuda.reset_peak_memory_stats()
S = einsum(Q, K, "b q d, b k d -> b q k") / math.sqrt(16)
P = torch.softmax(S, dim=-1)
O = einsum(P, V, "b q k, b k d -> b q d"); torch.cuda.synchronize()
print(f"torch.softmax 版前向 OK：峰值 {torch.cuda.max_memory_allocated() / GiB:.2f} GiB")
del Q, K, V, S, P, O; torch.cuda.empty_cache()

hw1 版前向 OOM：炸时已分配 24.0 GiB
torch.softmax 版前向 OK：峰值 16.05 GiB


### 4.2 `torch_compile`

(a) 同一份 sdpa 用 `torch.compile` 包一层，网格、预热、步数和 §4.1 完全一致，对比 fwd / bwd。
(b) 整个 Transformer `torch.compile(model)`，走第一篇 §2.1 的整模型 benchmark（`BenchConfig(compile=True)`），
三档 × forward / fwd_bwd / full，对比 eager。

预热是关键：`torch.compile` 的编译发生在**第一次调用**，秒级；第一篇 §2.1(c) 说的「eager 一步就稳、compile 留到 §4.2 看」就是这里。
先用 `warmup=0` 看首步开销，再决定正式表用几步预热。

In [ ]:
# (a) 编译版 attention：被测函数就是 sdpa 包一层 torch.compile，其余和 §4.1 一模一样
sdpa_compiled = torch.compile(sdpa)

df_c = attention_sweep(sdpa_compiled, D_MODELS, SEQ_LENS, f"{OUT}/attention_compiled.md")

# eager / compiled 并排 + 加速比
cmp = df[["d", "seq", "fwd_ms", "bwd_ms", "peak_fwd_gib", "status"]].merge(
    df_c[["d", "seq", "fwd_ms", "bwd_ms", "peak_fwd_gib", "status"]], on=["d", "seq"], suffixes=("_eager", "_compiled"))
cmp["fwd_speedup"] = cmp.fwd_ms_eager / cmp.fwd_ms_compiled
cmp["bwd_speedup"] = cmp.bwd_ms_eager / cmp.bwd_ms_compiled
cmp.to_json(f"{OUT}/attention_eager_vs_compiled.json", orient="records", indent=1)
cmp

In [ ]:
# (b-0) 首步开销：warmup=0，看 torch.compile 第一步花多久（编译时间），eager 对照
from benchmark import BenchConfig, run_model
first = []
for compile in (False, True):
    r = run_model(BenchConfig("small", 512, "full", warmup=0, steps=5, compile=compile))
    first.append(dict(compile=compile, first_ms=r.first_ms, rest_avg_ms=r.rest_avg_ms))
pd.DataFrame(first)

In [ ]:
# (b) 整个 Transformer：eager vs torch.compile(model)，三档 × 三模式（batch 4，seq 512，同第一篇 §2.1）
rows = []
for size in ("small", "medium", "large"):
    for mode in ("forward", "fwd_bwd", "full"):
        for compile in (False, True):
            r = run_model(BenchConfig(size, 512, mode, warmup=5, steps=10, compile=compile))
            rows.append(dict(size=size, mode=mode, compile=compile, avg_ms=r.avg_ms, std_ms=r.std_ms,
                             peak_mem_gib=r.peak_mem_gib, status=r.status))
model_cmp = pd.DataFrame(rows)
model_cmp.to_json(f"{OUT}/model_eager_vs_compiled.json", orient="records", indent=1)
model_cmp.pivot(index=["size", "mode"], columns="compile", values="avg_ms")